# midsphere SDK — advanced use cases

Production patterns built on the [quickstart](./midsphere_quickstart.ipynb): a reusable
streaming consumer, concurrent fan-out across tenants, per-user quota enforcement, a
human-in-the-loop approval policy, and batch cost/usage analytics.

Requires the backend running on `http://127.0.0.1:8010` and the agent runtime (E2B
sandbox) reachable. Runs here execute the real agent loop, so each takes ~15–30s.

## Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path("midsphere.env"))   # config written by dev-up.sh

import asyncio
from dataclasses import dataclass, field
from decimal import Decimal

from midsphere import Midsphere, MidsphereUserError

client = Midsphere(
    api_key=os.environ["MIDSPHERE_API_KEY"],
    base_url=os.environ.get("MIDSPHERE_BASE_URL", "http://localhost:8010"),
)

agent = (await client.agents.list())[0]
# A primary end user with generous controls for the demos below.
user = await client.end_users.ensure(external_user_id="adv|primary", display_name="Primary")
await client.end_users.set_controls(id=user.id, credits=500, concurrency_max=10, model_tier="pro")
print("agent:", agent.name, "| user:", user.id)

## 1. A reusable streaming consumer

`run.stream()` yields raw events; real apps want a **structured outcome**. This
`consume_run` helper routes each event type into a typed result — transcript, tool
calls, token/cost usage, and terminal status — so callers never touch the event loop.

In [ ]:
@dataclass
class RunResult:
    status: str = "unknown"          # FINISHED | FAILED | CANCELLED
    summary: str = ""
    steps: list[str] = field(default_factory=list)       # planner/agent step outputs
    tool_calls: list[tuple] = field(default_factory=list)  # (tool_name, arguments)
    tokens: int = 0
    cost: Decimal = Decimal("0")
    error: str | None = None
    event_seq: list[str] = field(default_factory=list)

    def _absorb_usage(self, usage: dict) -> None:
        self.tokens = usage.get("total_tokens", self.tokens)
        self.cost = Decimal(str(usage.get("cost_usd", self.cost)))


async def consume_run(run) -> RunResult:
    """Drive a run to its terminal event, folding the stream into a RunResult."""
    r = RunResult()
    async for ev in run.stream():
        r.event_seq.append(ev.type)
        p = ev.payload
        if ev.type == "step" and p.get("output"):
            r.steps.append(p["output"])
        elif ev.type == "tool_call":
            r.tool_calls.append((p.get("tool_name"), p.get("arguments")))
        elif ev.type == "usage_checkpoint":
            r._absorb_usage(p)
        elif ev.type == "run_completed":
            r.status, r.summary = "FINISHED", p.get("summary", "")
            r._absorb_usage(p.get("final_usage", {}))
        elif ev.type in ("run_failed", "run_cancelled"):
            r.status = "FAILED" if ev.type == "run_failed" else "CANCELLED"
            r.error = p.get("message") or p.get("reason")
            r._absorb_usage(p.get("final_usage", {}))
    return r

In [ ]:
run = await client.run(agent_id=agent.id, end_user_id=user.id,
                       prompt="In one short sentence, why is the sky blue?")
res = await consume_run(run)
print(f"status   : {res.status}")
print(f"usage    : {res.tokens} tokens  ${res.cost}")
print(f"summary  : {res.summary[:200]}")
print(f"events   : {res.event_seq}")

## 2. Concurrent fan-out across tenants

A multi-tenant app runs the **same agent for many end users at once**. `asyncio.gather`
fans out the runs; each streams independently through `consume_run`; then we aggregate
usage into a billing-style report. Concurrency is enforced *per end user*, so distinct
users run in parallel without tripping each other's caps.

In [ ]:
async def onboard(ext_id: str, name: str) -> object:
    u = await client.end_users.ensure(external_user_id=ext_id, display_name=name)
    await client.end_users.set_controls(id=u.id, credits=200, concurrency_max=5, model_tier="pro")
    return u

tenants = await asyncio.gather(
    onboard("adv|alice", "Alice"),
    onboard("adv|bob", "Bob"),
    onboard("adv|carol", "Carol"),
)

async def run_for(u, prompt: str):
    run = await client.run(agent_id=agent.id, end_user_id=u.id, prompt=prompt)
    return u.display_name, await consume_run(run)

prompts = ["What is 12 times 12?", "Name one primary color.", "Capital of France, one word."]
results = await asyncio.gather(*(run_for(u, p) for u, p in zip(tenants, prompts)))

In [ ]:
print(f"{'user':8} {'status':9} {'tokens':>7} {'cost':>10}   summary")
print("-" * 70)
total_tokens, total_cost = 0, Decimal("0")
for name, r in results:
    total_tokens += r.tokens
    total_cost += r.cost
    print(f"{name:8} {r.status:9} {r.tokens:7d} ${r.cost:>9.5f}   {r.summary[:34]}")
print("-" * 70)
print(f"{'TOTAL':8} {'':9} {total_tokens:7d} ${total_cost:>9.5f}")

## 3. Per-user quota enforcement

Each end user carries controls — credits, concurrency, model tier, allowed tools. The
platform enforces them *before* a run starts and surfaces a `MidsphereUserError` with a
**stable code** you can branch on. Here we zero out a user's credits and watch the run
get refused.

In [ ]:
broke = await client.end_users.ensure(external_user_id="adv|tightwad", display_name="Tightwad")
await client.end_users.set_controls(id=broke.id, credits=0, concurrency_max=1, model_tier="free")

try:
    await client.run(agent_id=agent.id, end_user_id=broke.id, prompt="hello")
    print("run started (unexpected)")
except MidsphereUserError as e:
    print(f"refused: {e}  (code={e.code})")
    if e.has("INSUFFICIENT_CREDITS"):
        print("  -> branch: prompt the end user to top up before retrying")
    elif e.has("CONCURRENCY_CAP"):
        print("  -> branch: queue the run and retry when a slot frees")
    elif e.has("MODEL_TIER_RESTRICTED"):
        print("  -> branch: offer an upgrade")

## 4. Human-in-the-loop approval policy

When an agent needs sign-off it emits `hitl_requested`; you answer with
`run.respond(request_id, decision)`. This consumer applies an **automated policy** —
deny anything matching risky keywords, approve the rest — while still streaming the run
to completion. (The demo agent may not trigger HITL; the handler is production-ready
regardless.)

In [ ]:
DENY_KEYWORDS = ("delete", "wire transfer", "drop table", "production")

async def run_with_hitl_policy(prompt: str):
    run = await client.run(agent_id=agent.id, end_user_id=user.id, prompt=prompt)
    decisions: list[tuple[str, str]] = []
    async for ev in run.stream():
        if ev.type == "hitl_requested":
            req = ev.payload
            risky = any(k in req["prompt"].lower() for k in DENY_KEYWORDS)
            decision = "deny" if risky else "approve"
            await run.respond(req["request_id"], decision)
            decisions.append((req["prompt"][:40], decision))
        elif ev.type in ("run_completed", "run_failed", "run_cancelled"):
            return ev.type, decisions
    return "incomplete", decisions

terminal, decisions = await run_with_hitl_policy("Summarize our refund policy in one line.")
print("terminal event:", terminal)
print("HITL decisions:", decisions or "none requested")

## 5. Batch analytics + per-run debug consoles

Run a batch, capture each run's **live debug-console URL** (the in-sandbox Astro UI on
port 4321) for inspection, and roll usage up into a cost report.

In [ ]:
batch = [
    ("invoice-bot", "What is 7 squared?"),
    ("speller",     "Spell 'midsphere' backwards."),
]

report = []
for label, prompt in batch:
    run = await client.run(agent_id=agent.id, end_user_id=user.id, prompt=prompt)
    console = run.debug_console_url            # inspect this run live while it streams
    res = await consume_run(run)
    report.append((label, res, console))

print(f"{'job':12} {'status':9} {'tokens':>7} {'cost':>9}   debug console")
print("-" * 86)
for label, r, console in report:
    print(f"{label:12} {r.status:9} {r.tokens:7d} ${r.cost:>8.5f}   {console}")

n = len(report)
avg_cost = sum((r.cost for _, r, _ in report), Decimal("0")) / n
avg_tokens = sum(r.tokens for _, r, _ in report) // n
print("-" * 86)
print(f"averages: {avg_tokens} tokens/run, ${avg_cost:.5f}/run across {n} runs")

## Cleanup

In [ ]:
await client.aclose()
print("done")